# Master ML Dataset Creation Pipeline (Pure UTC, 2021–2024)

This notebook assembles, cleans, imputes, and merges all market and fundamental data streams for the German electricity bidding zone and its four Transmission System Operator (TSO) control areas onto a uniform 15-minute Pure UTC (`+00:00` / `Z`) time grid.

### Methodological & Regulatory Standards:
1. **Pure UTC Adherence:** All timestamps across all streams are anchored strictly to Pure UTC (`+00:00`).
2. **TSO Delivery Area Mapping:** Four German TSOs mapped 1:1 to EPEX delivery areas: `DE1` (TransnetBW), `DE2` (Amprion), `DE3` (TenneT), `DE4` (50Hertz), `DE` (National aggregate).
3. **Canonical Nomenclature:** All rolling price and volume metrics adhere to standard **`VWAP`** naming (e.g. `VWAP_90to105`, `DE1_VWAP_0to30`).
4. **Hierarchical Imputation (Variante B):** Continuous price trajectories impute missing values via contemporaneous national benchmarks, preceding trading windows, and Intraday Auction prices.
5. **No Data Loss & 0 NaNs:** Full 114,052 quarter-hour grid preserved with zero missing values across all 962 columns.
6. **Feature Manifest:** Ordered canonically via `features.csv`.


In [ ]:
# Environment & Configuration Setup
import os
import sys
import glob
import time
import pandas as pd
import numpy as np

# Project Paths
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "Data")
FEATURES_CSV = os.path.join(BASE_DIR, "features.csv")
OUTPUT_CSV = os.path.join(DATA_DIR, "master_dataset_2021_2024.csv")

# Ensure local modules are accessible
sys.path.insert(0, os.path.join(DATA_DIR, "DA Auction Spot Prices"))
sys.path.insert(0, os.path.join(DATA_DIR, "ID Auction Spot Prices"))

# Canonical TSO to EPEX Delivery Area Mapping (per AGENTS.md)
TSO_TO_DELIVERY_AREA = {
    "TransnetBW": "DE1",
    "Amprion": "DE2",
    "TenneT": "DE3",
    "TenneT TSO": "DE3",
    "50Hertz": "DE4",
    "Deutschland": "DE",
}

# Pure UTC Date Spine Range
# 2021-09-30 22:00:00 UTC = 2021-10-01 00:00 CEST (Q4 2021 delivery start)
# 2024-12-31 22:45:00 UTC = 2024-12-31 23:45 CET  (last 2024 delivery interval start)
START_UTC = "2021-09-30 22:00:00"
END_UTC = "2024-12-31 22:45:00"
# 7-day look-ahead buffer for multi-day lags (672 quarter-hours prior to START_UTC)
BUFFER_START_UTC = "2021-09-23 22:00:00"
FREQ = "15min"

print(f"Base Directory:   {BASE_DIR}")
print(f"Target Output:     {OUTPUT_CSV}")
print(f"Delivery Horizon:  {START_UTC} to {END_UTC}")


## Step 1: Pure UTC Date Spine Creation
Generate a continuous 15-minute time grid in Pure UTC (`+00:00`) with a 7-day lookback buffer to prevent edge-case NaNs in multi-day lags.


In [ ]:
# 1. Date Spine Creation (with 7-day lag buffer)
date_range = pd.date_range(start=BUFFER_START_UTC, end=END_UTC, freq=FREQ, tz="UTC")
master_df = pd.DataFrame(date_range, columns=["Date"])

print(f"Created Pure UTC date spine with {len(master_df):,} quarter-hours (including 7-day lag buffer).")
print(f"Spine Start: {master_df['Date'].iloc[0]}")
print(f"Spine End:   {master_df['Date'].iloc[-1]}")
master_df.head(3)


## Step 2: Activated Balancing Power (SRL & MRL)
Loads operational balancing energy files from `Data/Balancing Energy/`:
- **SRL (aFRR - Automatic Frequency Restoration Reserve):** 2021–2024
- **MRL (mFRR - Manual Frequency Restoration Reserve):** 2021–2024
- Strips obsolete MOL-Abweichung and maps TSOs strictly to `DE1`–`DE4` and `DE`.


In [ ]:
# 2. Load Balancing Energy (SRL / aFRR & MRL / mFRR)
be_dir = os.path.join(DATA_DIR, "Balancing Energy")
res_dfs = []

for res_type in ["SRL", "MRL"]:
    files = sorted(glob.glob(os.path.join(be_dir, res_type, "*.csv")))
    type_dfs = []
    for file_path in files:
        df = pd.read_csv(file_path, sep=";", decimal=".")
        dt_str = df["Datum"].astype(str).str.replace(".", "-", regex=False) + " " + df["von"].astype(str)
        df["Date"] = pd.to_datetime(dt_str, format="%d-%m-%Y %H:%M", utc=True)

        drop_cols = [c for c in ["Datum", "Zeitzone", "von", "bis", "Einheit", "MOL-Abweichung"] if c in df.columns]
        df = df.drop(columns=drop_cols)

        rename_map = {}
        for col in df.columns:
            if col == "Date":
                continue
            is_pos = "(Positiv)" in col
            sign = "positive" if is_pos else "negative"
            area_raw = col.replace(" (Positiv)", "").replace(" (Negativ)", "").strip()
            zone = TSO_TO_DELIVERY_AREA.get(area_raw, area_raw)
            rename_map[col] = f"{zone}_{res_type}_{sign}"
        df = df.rename(columns=rename_map)
        type_dfs.append(df)

    combined_type = pd.concat(type_dfs, ignore_index=True).sort_values("Date").drop_duplicates(subset=["Date"])
    res_dfs.append(combined_type)

be_df = res_dfs[0]
for other in res_dfs[1:]:
    be_df = be_df.merge(other, on="Date", how="outer")

master_df = master_df.merge(be_df, on="Date", how="left")
print(f"Merged Balancing Energy: {len(be_df.columns) - 1} reserve activation features.")
master_df.filter(regex=r"SRL|MRL").head(3)


## Step 3: Energy-Charts Fundamentals (Load, Solar, Wind, Cross-Border Flows)
Loads fundamental time series across 5 categories:
- **LOAD:** Actual total load & Day-Ahead forecast
- **SOLAR:** Actual solar PV feed-in & Day-Ahead forecast
- **ONSHORE:** Actual wind onshore feed-in & Day-Ahead forecast
- **OFFSHORE:** Actual wind offshore feed-in & Day-Ahead forecast
- **CROSS_BORDER:** Actual cross-border electricity trading flows (*Grenzüberschreitender Stromhandel*)
- Maps control areas to canonical EPEX delivery area codes (`DE1`–`DE4`, `DE`).

In [ ]:
# 3. Load Fundamentals (Load, Solar, Wind Onshore & Offshore, Cross-Border Flows)
fund_base = os.path.join(DATA_DIR, "Fundamentals")
categories = [
    ("LOAD", "load"),
    ("SOLAR", "solar"),
    ("ONSHORE", "wind_onshore"),
    ("OFFSHORE", "wind_offshore")
]

all_cat_dfs = []
for folder_name, prefix in categories:
    cat_dir = os.path.join(fund_base, folder_name)
    files = sorted(glob.glob(os.path.join(cat_dir, "*.csv")))
    cat_dfs = []

    for file_path in files:
        df = pd.read_csv(file_path, skiprows=[1])
        df = df.rename(columns={"Date (UTC)": "Date"})
        df["Date"] = pd.to_datetime(df["Date"], utc=True)

        rename_dict = {}
        for col in df.columns:
            if col == "Date":
                continue
            is_forecast = "forecast" in col.lower()
            clean_col = col.replace(" [MW] Realisierte Erzeugung", "")
            clean_col = clean_col.replace(" [MW] Realisierter Stromverbrauch", "")
            clean_col = clean_col.replace(" [MW] Vorhersage", "")
            clean_col = clean_col.replace(" [MW] Day-ahead", "")
            clean_col = clean_col.replace(" [MW]", "").strip()

            zone = TSO_TO_DELIVERY_AREA.get(clean_col, clean_col)
            suffix = "forecast" if is_forecast else "actual"
            rename_dict[col] = f"{zone}_{prefix}_{suffix}"

        df = df.rename(columns=rename_dict)
        cat_dfs.append(df)

    combined_cat = pd.concat(cat_dfs, ignore_index=True).sort_values("Date").drop_duplicates(subset=["Date"])

    # Forecast errors (diff = actual - forecast)
    for zone in ["DE1", "DE2", "DE3", "DE4", "DE"]:
        act_col = f"{zone}_{prefix}_actual"
        fc_col = f"{zone}_{prefix}_forecast"
        diff_col = f"{zone}_{prefix}_diff"
        if act_col in combined_cat.columns and fc_col in combined_cat.columns:
            combined_cat[diff_col] = combined_cat[act_col] - combined_cat[fc_col]

    all_cat_dfs.append(combined_cat)

# Load Cross-Border Flows (Germany)
cb_dir = os.path.join(fund_base, "CROSS_BORDER")
if os.path.isdir(cb_dir):
    cb_files = sorted(glob.glob(os.path.join(cb_dir, "*.csv")))
    cb_dfs = []
    for f in cb_files:
        df = pd.read_csv(f, skiprows=[1])
        df = df.rename(columns={"Date (UTC)": "Date"})
        df["Date"] = pd.to_datetime(df["Date"], utc=True)
        val_col = [c for c in df.columns if c != "Date"][0]
        cb_dfs.append(pd.DataFrame({
            "Date": df["Date"],
            "DE_cross_border_trading": pd.to_numeric(df[val_col], errors="coerce")
        }))
    if cb_dfs:
        combined_cb = pd.concat(cb_dfs, ignore_index=True).sort_values("Date").drop_duplicates(subset=["Date"])
        all_cat_dfs.append(combined_cb)

fund_df = all_cat_dfs[0]
for other in all_cat_dfs[1:]:
    fund_df = fund_df.merge(other, on="Date", how="outer")

# Canonical aliases for legacy model compatibility
if "DE_load_actual" in fund_df.columns:
    fund_df["Load"] = fund_df["DE_load_actual"]
if "DE_solar_diff" in fund_df.columns:
    fund_df["prediction_error_solar"] = fund_df["DE_solar_diff"]
if "DE_wind_onshore_diff" in fund_df.columns:
    fund_df["prediction_error_onshore"] = fund_df["DE_wind_onshore_diff"]
if "DE_wind_offshore_diff" in fund_df.columns:
    fund_df["prediction_error_offshore"] = fund_df["DE_wind_offshore_diff"]
if "DE_cross_border_trading" in fund_df.columns:
    fund_df["cross_border_trading"] = fund_df["DE_cross_border_trading"]

master_df = master_df.merge(fund_df, on="Date", how="left")
print(f"Merged Fundamentals: {len(fund_df.columns) - 1} generation, demand, forecast & cross-border features.")
master_df.filter(regex=r"load|solar|wind|cross_border").head(3)


## Step 4: Day-Ahead & Intraday Auction Spot Prices
- **Day-Ahead Hourly Auction (`Dayahead_Auction_hourly`):** Cleared at 12:00 D-1, mapped continuously to 15-minute intervals.
- **Intraday 15-Minute Auction (`Intraday Auction Price`):** Cleared at 15:00 D-1 (combining 15-Call and Pan-European IDA1).


In [ ]:
# 4. Load Auction Spot Prices (DA and ID)
from process_da_auction_prices import process_da_auction_spot_prices
from process_id_auction_prices import process_id_auction_spot_prices

da_df = process_da_auction_spot_prices(os.path.join(DATA_DIR, "DA Auction Spot Prices"))
id_df = process_id_auction_spot_prices(os.path.join(DATA_DIR, "ID Auction Spot Prices"))

if "Intraday_Auction_Price" in id_df.columns:
    id_df = id_df.rename(columns={"Intraday_Auction_Price": "Intraday Auction Price"})

auctions_df = da_df.merge(id_df, on="Date", how="outer").sort_values("Date")
master_df = master_df.merge(auctions_df, on="Date", how="left")
print("Merged Auction Spot Prices: Dayahead_Auction_hourly & Intraday Auction Price.")
master_df[["Date", "Dayahead_Auction_hourly", "Intraday Auction Price"]].head(3)


## Step 5: EPEX Intraday Continuous Transactions
Loads rolling continuous metrics across regional zones (`DE1`, `DE2`, `DE3`, `DE4`) and the national aggregate (`DE`):
- **`VWAP`:** Volume-Weighted Average Prices across 15-minute rolling lead-time windows.
- **`total_volume` & `total_trades`:** Trading activity per window.
- **`id_balance_to_`:** Inter-zonal and cross-border physical continuous commercial flows.


In [ ]:
# 5. Load EPEX Intraday Continuous Transactions
cont_dir = os.path.join(DATA_DIR, "EPEX Intraday Continuous")
combined_cont = None

for area in ["DE1", "DE2", "DE3", "DE4", "DE"]:
    files = sorted(glob.glob(os.path.join(cont_dir, f"ID_DelArea_{area}_*.csv")))
    if not files:
        continue

    area_dfs = []
    for file_path in files:
        df = pd.read_csv(file_path)

        if area == "DE":
            keep_cols = [
                c for c in df.columns
                if c.startswith("VWAP_") and not any(x in c for x in ["total_volume", "total_trades", "upper", "lower"])
            ]
        else:
            keep_cols = [
                c for c in df.columns
                if c != "DeliveryStart"
                and not any(x in c for x in ["upper", "lower"])
                and not c.startswith(f"id_balance_to_{area}_")
            ]

        df_sub = df[["DeliveryStart"] + keep_cols].copy()
        df_sub["Date"] = pd.to_datetime(df_sub["DeliveryStart"], utc=True)
        df_sub = df_sub.drop(columns=["DeliveryStart"])

        for c in keep_cols:
            df_sub[c] = pd.to_numeric(df_sub[c], errors="coerce")
        area_dfs.append(df_sub)

    area_df = pd.concat(area_dfs, ignore_index=True).sort_values("Date").drop_duplicates(subset=["Date"])

    if area != "DE":
        rename_dict = {col: f"{area}_{col}" for col in area_df.columns if col != "Date"}
        area_df = area_df.rename(columns=rename_dict)

    if combined_cont is None:
        combined_cont = area_df
    else:
        combined_cont = combined_cont.merge(area_df, on="Date", how="outer")

master_df = master_df.merge(combined_cont, on="Date", how="left")
print(f"Merged EPEX Intraday Continuous: {len(combined_cont.columns) - 1} continuous metrics.")
master_df.filter(regex=r"VWAP_90to105").head(3)


## Step 6: Calendar, Hourly & Quarter-Hour Delivery Indicators
Generates granular wall-clock delivery indicators (Europe/Berlin):
- **`Weekday_1` to `Weekday_7`:** Day of the week (1 = Monday, 7 = Sunday).
- **`Hour_0` to `Hour_23`:** 24 hourly delivery block dummies.
- **`Quarter_1` to `Quarter_4`:** 4 sub-hourly quarter dummies (:00, :15, :30, :45).


In [ ]:
# 6. Add Calendar, Hourly & Quarter Indicators
local_date = master_df["Date"].dt.tz_convert("Europe/Berlin")
weekday = local_date.dt.dayofweek + 1
hour = local_date.dt.hour
minute = local_date.dt.minute
quarter = (minute // 15) + 1

dummies = {}
for i in range(1, 8):
    dummies[f"Weekday_{i}"] = (weekday == i).astype(int)
for h in range(24):
    dummies[f"Hour_{h}"] = (hour == h).astype(int)
for q in range(1, 5):
    dummies[f"Quarter_{q}"] = (quarter == q).astype(int)

dummy_df = pd.DataFrame(dummies, index=master_df.index)
master_df = pd.concat([master_df, dummy_df], axis=1)

print(f"Generated {len(dummy_df.columns)} calendar, hourly, and quarter-hour dummy indicators.")
master_df.filter(regex=r"Weekday_|Hour_|Quarter_").head(3)


## Step 7: Hierarchical Missing Value Imputation (Variante B)
Addresses illiquid trading windows where no transactions took place:
- **National Trajectory (`DE`):** Leftmost window (`VWAP_345to360`) falls back to contemporaneous `Intraday Auction Price`. Subsequent windows forward-fill from preceding window.
- **Regional Trajectories (`DE1`–`DE4`):** Prio 0 (own trade) -> Prio 1 (contemporaneous national volume-weighted price `VWAP_{w}`).
- **Non-Price Metrics:** Missing traded volumes, counts, or flow balances filled with 0 (no activity).


In [ ]:
# 7. Hierarchical Imputation of Continuous Trading Trajectories (Variante B)
windows_15m = [
    "345to360", "330to345", "315to330", "300to315",
    "285to300", "270to285", "255to270", "240to255",
    "225to240", "210to225", "195to210", "180to195",
    "165to180", "150to165", "135to150", "120to135",
    "105to120", "90to105", "75to90", "60to75",
    "45to60", "30to45", "15to30", "0to15"
]

# 1. National DE Trajectory
if "VWAP_345to360" in master_df.columns:
    master_df["VWAP_345to360"] = master_df["VWAP_345to360"].fillna(master_df["Intraday Auction Price"])
    master_df["VWAP_345to360"] = master_df["VWAP_345to360"].fillna(master_df["Dayahead_Auction_hourly"])

for i in range(1, len(windows_15m)):
    curr_w = windows_15m[i]
    prev_w = windows_15m[i - 1]
    curr_col = f"VWAP_{curr_w}"
    prev_col = f"VWAP_{prev_w}"
    if curr_col in master_df.columns and prev_col in master_df.columns:
        master_df[curr_col] = master_df[curr_col].fillna(master_df[prev_col])

if "VWAP_0to30" in master_df.columns:
    master_df["VWAP_0to30"] = master_df["VWAP_0to30"].fillna(master_df.get("VWAP_0to15", np.nan)).fillna(master_df.get("VWAP_15to30", np.nan))

# 2. Regional Delivery Areas DE1-DE4
all_windows = windows_15m + (["0to30"] if "VWAP_0to30" in master_df.columns else [])
for area in ["DE1", "DE2", "DE3", "DE4"]:
    for w in all_windows:
        reg_col = f"{area}_VWAP_{w}"
        nat_col = f"VWAP_{w}"
        if reg_col in master_df.columns and nat_col in master_df.columns:
            master_df[reg_col] = master_df[reg_col].fillna(master_df[nat_col])

# 3. Non-price trading features: fill NaN with 0 (no trades / 0 MW / 0 EUR)
non_price_cols = [
    c for c in master_df.columns
    if any(k in c for k in ["total_volume", "total_trades", "balance_to_"])
]
for c in non_price_cols:
    master_df[c] = master_df[c].fillna(0)

vwap_cols = [c for c in master_df.columns if "VWAP_" in c and not any(x in c for x in ["total_volume", "total_trades", "upper", "lower", "lag_"])]
print(f"Hierarchical imputation completed. Remaining NaNs across all {len(vwap_cols)} VWAP columns: {master_df[vwap_cols].isna().sum().sum()}")


## Step 8: Multi-Day Time Lags & Date Spine Trimming
- **Multi-Day Lags:** Generates $d-1$ (24h = 96 steps), $d-2$ (48h = 192 steps), and $d-7$ (168h = 672 steps) shifts for key auction and continuous benchmarks.
- **Trimming:** Trims the 7-day lookback buffer to retain strictly the canonical delivery horizon (`START_UTC` to `END_UTC`), resulting in zero NaNs at the beginning of the series.


In [ ]:
# 8. Multi-Day Lags & Lag Buffer Trimming
lag_shifts = {
    "lag_24h": 96,    # d-1 (24 hours)
    "lag_48h": 192,   # d-2 (48 hours)
    "lag_168h": 672   # d-7 (7 days / 168 hours)
}

potential_lag_targets = [
    "VWAP_90to105",
    "Dayahead_Auction_hourly",
    "Intraday Auction Price",
    "DE1_VWAP_90to105",
    "DE2_VWAP_90to105",
    "DE3_VWAP_90to105",
    "DE4_VWAP_90to105"
]
cols_to_lag = [c for c in potential_lag_targets if c in master_df.columns]

lagged_dict = {}
for col in cols_to_lag:
    for lag_name, shift_val in lag_shifts.items():
        lagged_dict[f"{col}_{lag_name}"] = master_df[col].shift(shift_val)

lag_df = pd.DataFrame(lagged_dict, index=master_df.index)
master_df = pd.concat([master_df, lag_df], axis=1)

# Trim lag buffer to the canonical delivery horizon [START_UTC, END_UTC]
master_df = master_df[(master_df["Date"] >= START_UTC) & (master_df["Date"] <= END_UTC)].copy()

print(f"Generated {len(lag_df.columns)} multi-day lag features.")
print(f"Trimmed master dataset: {len(master_df):,} quarter-hours ({master_df['Date'].iloc[0]} to {master_df['Date'].iloc[-1]}).")
master_df[["Date", "VWAP_90to105", "VWAP_90to105_lag_24h", "VWAP_90to105_lag_168h"]].head(3)


## Step 9: Alignment with `features.csv` & Export
Sorts columns to match the canonical feature catalog (`features.csv`) and exports the final master training dataset.


In [ ]:
# 9. Align Column Ordering via features.csv & Export
if os.path.exists(FEATURES_CSV):
    f_df = pd.read_csv(FEATURES_CSV)
    canonical_order = f_df["name"].dropna().astype(str).str.strip().tolist()

    matched_cols = [c for c in canonical_order if c in master_df.columns]
    unmatched_cols = [c for c in master_df.columns if c not in matched_cols]

    master_df = master_df[matched_cols + unmatched_cols]
    print(f"Aligned with features.csv: {len(matched_cols)} matched, {len(unmatched_cols)} appended.")
else:
    print("features.csv not found; preserving existing order.")

print(f"Saving master dataset to '{OUTPUT_CSV}' ...")
master_df.to_csv(OUTPUT_CSV, index=False)
file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
print(f"Master file saved successfully! Size: {file_size_mb:.2f} MB, Rows: {len(master_df):,}, Columns: {len(master_df.columns)}")


## Step 10: Final Quality Control & Summary Statistics
Verifies that all columns have exactly 0 NaNs and inspects regional target variables.


In [ ]:
# 10. Quality Verification & Inspection
targets = ["DE1_VWAP_0to30", "DE2_VWAP_0to30", "DE3_VWAP_0to30", "DE4_VWAP_0to30"]
print("Regional Target Variables Summary Statistics:")
display(master_df[targets].describe().T)

total_nans = master_df.isna().sum().sum()
print(f"\nTotal Missing Values (NaNs): {total_nans}")
print(f"Timestamp Duplicates:        {master_df['Date'].duplicated().sum()}")
print(f"Total Rows:                  {len(master_df):,}")
print(f"Total Columns:               {len(master_df.columns)}")
print(f"Memory Usage:                {master_df.memory_usage().sum() / (1024 * 1024):.2f} MB")
assert total_nans == 0, f"Error: Dataset contains {total_nans} unexpected NaNs!"
